# Nemotron finetuning pipeline — flat shuffle (scratch / continue)

Drops the competence-based curriculum. Trains a flat-shuffled pool of
cryptarithm examples (`examples_3/4/5`) and non-cryptarithm replay.
No math replay.

| `EXPERIMENT` | LoRA init | Crypt data | Replay data | Est. steps |
|---|---|---|---|---|
| `"scratch"` | fresh | combined\_1k (3k, x1) | no\_cryptarithm (7149 w/ dups, x1) | ~317 |
| `"continue"` | 0.86 adapter | combined\_2k (6k, x1) | no\_cryptarithm\_dedup (6106, x1) | ~378 |

Estimated time @ ~1 min/step: **scratch ~5h20m**, **continue ~6h20m** (lower bound;
crypt CoTs may be longer than non-crypt -> up to +25%).

In [ ]:
# ── Shared config ─────────────────────────────────────────────────────
EXPERIMENT = "scratch"  # "scratch" | "continue"
# scratch : fresh LoRA,        combined_1k (3k crypt, examples_3/4/5) + train_corpus_no_cryptarithm (7149 w/ dups)
# continue: load 0.86 adapter, combined_2k (6k crypt, 2k/category)   + train_corpus_no_cryptarithm_dedup (6106)

LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 1000        # cap; actual steps = len(pool) // BATCH_SIZE (flat single-pass)
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE = 2e-4
# RESET_WEIGHTS is derived from EXPERIMENT inside run_training()
IN_PROJ_ONLY = False
MOE_TIE_WEIGHTS = True  # tie one side of MoE expert LoRA across all 128 experts (Tinker-style)
SKIP_TRAINING = False   # if True, save fresh zero-delta LoRA (base baseline)

KAGGLE_DATASET = "huikang/nemotron-data"
MINUTES = 60

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj", "in_proj", "out_proj",
    "lm_head",
]

In [ ]:
# ── Flat-schedule + reproducibility config ────────────────────────────
# No curriculum: crypt + replay examples are pooled and flat-shuffled.
# CRYPT_EPOCHS / REPLAY_EPOCHS replicate each dataset before shuffling:
#   1 = single-pass (default), 2 = each row seen twice, etc.
CRYPT_EPOCHS  = 1
REPLAY_EPOCHS = 1
SEED          = 0   # flat-shuffle seed (deterministic order)
REPLAY_SUBSAMPLE = 4000  # subsample replay rows before pooling (None = use all)

# Boxed-answer instruction — must match inference harness + mask_clues anchors.
BOXED_INSTRUCTION = (
    "Please put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ
IS_MODAL_LAUNCHER = not IS_KAGGLE and not IS_MODAL_WORKER

In [ ]:
# ── Env-specific install (Kaggle only; Modal image has packages pre-installed) ──
if IS_KAGGLE:
    import subprocess

    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True, check=True,
    )
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                ["pip", "install", "-q", "--no-index", "--find-links", _wd,
                 "protobuf==6.33.5", "sentencepiece", "safetensors", "huggingface_hub"],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)

In [ ]:
# ── Data helpers: parquet -> tokens+mask; flat schedule ───────────────

def _rearrange_think(full_text, cot, answer):
    EMPTY_THINK = "<think></think>"
    assert EMPTY_THINK in full_text, repr(full_text[-80:])
    idx    = full_text.rindex(EMPTY_THINK)
    before = full_text[:idx]
    after  = full_text[idx + len(EMPTY_THINK):]
    tail   = after[after.index("<|im_end|>"):]
    return before + "<think>\n" + cot + " \n</think>\n\\boxed{" + answer + "}\n" + tail


def render_and_mask(tok, prompt, cot, answer, clues, max_seq_len, full_fixed=None):
    if full_fixed is None:
        user_content = f"{prompt}\n{BOXED_INSTRUCTION}"
        full_turn    = [{"role": "user", "content": user_content},
                        {"role": "assistant", "content": cot}]
        full_raw     = tok.apply_chat_template(full_turn, tokenize=False, enable_thinking=False)
        full_fixed   = _rearrange_think(full_raw, cot, answer)

    def _prefix_match_len(full_ids, prefix_ids):
        n = 0
        for a, b in zip(full_ids, prefix_ids):
            if a != b: break
            n += 1
        return n

    full_ids = tok.encode(full_fixed)
    mask     = [1] * len(full_ids)
    for cl in clues:
        b = full_fixed.find(cl["mask_begin_clue"])
        e = full_fixed.find(cl["mask_end_clue"])
        assert b != -1 and e != -1, f"mask clue not found in rendered text: {cl!r}"
        tok_start = _prefix_match_len(full_ids, tok.encode(full_fixed[:b]))
        tok_end   = _prefix_match_len(full_ids, tok.encode(full_fixed[:e + len(cl["mask_end_clue"])]))
        for i in range(tok_start, tok_end):
            mask[i] = 0
    if len(full_ids) > max_seq_len:
        full_ids = full_ids[:max_seq_len]
        mask     = mask[:max_seq_len]
    return full_ids, mask


def build_examples_from_parquet(tok, path, source, max_seq_len):
    import pandas as pd

    df = pd.read_parquet(path)
    if "cot" not in df.columns and "cot_1_5" in df.columns:
        df = df.rename(columns={"cot_1_5": "cot"})
    required = {"problem_id", "prompt", "answer", "cot", "mask_clues"}
    assert required <= set(df.columns), \
        f"missing columns in {path}: {required - set(df.columns)}"

    out = {}
    for row in df.itertuples(index=False):
        ft = getattr(row, "full_text", None)
        ft = ft if isinstance(ft, str) and ft else None
        ids, mask = render_and_mask(tok, row.prompt, row.cot, row.answer,
                                    list(row.mask_clues), max_seq_len, full_fixed=ft)
        if not any(mask):
            continue
        pid = row.problem_id if source == "crypt" else f"replay::{row.problem_id}"
        out[pid] = {
            "problem_id": pid,
            "tokens":  ids[:-1],
            "targets": ids[1:],
            "weights": [float(m) for m in mask[1:]],
            "source":  source,
            "bucket":  None,
        }
    return out


def build_flat_schedule(example_ids, batch_size, num_steps_cap, seed):
    import random

    rng = random.Random(seed)
    ids = list(example_ids)
    rng.shuffle(ids)
    max_steps = len(ids) // batch_size
    num_steps = min(num_steps_cap, max_steps)
    if num_steps < num_steps_cap:
        print(
            f"  NOTE: NUM_STEPS={num_steps_cap} capped to {num_steps} "
            f"({len(ids)} pool // {batch_size} batch_size)"
        )
    return [ids[i * batch_size : (i + 1) * batch_size] for i in range(num_steps)]

In [ ]:
def run_training() -> list[str]:
    import gc, json, math, random, subprocess, sys, time

    from unsloth import FastLanguageModel
    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear

    # ── Derive RESET_WEIGHTS from EXPERIMENT ──────────────────────────
    RESET_WEIGHTS = (EXPERIMENT == "scratch")
    print(f"EXPERIMENT={EXPERIMENT!r}  RESET_WEIGHTS={RESET_WEIGHTS}")

    # ── Env-specific paths ────────────────────────────────────────────
    if IS_KAGGLE:
        import kagglehub
        MODEL_PATH  = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
        ADAPTER_SRC = "/kaggle/input/models/zosov07/nemotron-lora-baseline-0-86/transformers/default/1/submission"
        if EXPERIMENT == "scratch":
            # combined_1k train-ready (3k rows, examples_3/4/5, with cot_1_5 + mask_clues)
            # TODO: update slug after uploading combined_1k_train_ready.parquet to Kaggle
            CRYPT_PARQUET  = "/kaggle/input/datasets/zosov07/training-data-v3-flat/combined_1k_train_ready.parquet"
            REPLAY_PARQUET = "/kaggle/input/datasets/zosov07/training-data-v3-flat/train_corpus_no_cryptarithm.parquet"
        else:  # continue
            # combined_2k train-ready (6k rows, 2k x examples_3/4/5, with cot_1_5 + mask_clues)
            # TODO: update slug after uploading combined_2k_train_ready.parquet to Kaggle
            CRYPT_PARQUET  = "/kaggle/input/datasets/zosov07/training-data-v3-flat/combined_2k_train_ready.parquet"
            REPLAY_PARQUET = "/kaggle/input/datasets/zosov07/training-data-v3-flat/train_corpus_no_cryptarithm_dedup.parquet"
    else:  # IS_MODAL_WORKER
        MODEL_PATH  = "unsloth/Nemotron-3-Nano-30B-A3B"
        ADAPTER_SRC = "/merged/weights"
        OUTPUT_DIR  = "/output/weights"
        if EXPERIMENT == "scratch":
            CRYPT_PARQUET  = "/data/combined_1k_train_ready.parquet"
            REPLAY_PARQUET = "/data/train_corpus_no_cryptarithm.parquet"
        else:  # continue
            CRYPT_PARQUET  = "/data/combined_2k_train_ready.parquet"
            REPLAY_PARQUET = "/data/train_corpus_no_cryptarithm_dedup.parquet"

    # ── GPU + kernel sanity check ─────────────────────────────────────
    import causal_conv1d, mamba_ssm
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0]*10+cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if IS_MODAL_WORKER:
        assert cc == (12, 0), f"Expected sm_120 (RTX PRO 6000), got sm_{cc[0]*10+cc[1]}"
    from causal_conv1d import causal_conv1d_fn
    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4,   device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    if IS_MODAL_WORKER:
        import shutil as _sh
        hf_modules = os.path.join(os.environ.get("HF_HOME", "/root/.cache/huggingface"), "modules")
        if os.path.exists(hf_modules):
            _sh.rmtree(hf_modules)

    # ── Load corpus (flat pool, no curriculum, no math replay) ────────
    examples_by_id: dict[str, dict] = {}
    schedule: list[list[str]] = []

    if SKIP_TRAINING:
        print("SKIP_TRAINING=True — skipping corpus load (baseline no-op adapter run)")
    else:
        from transformers import AutoTokenizer
        tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

        crypt  = build_examples_from_parquet(tok, CRYPT_PARQUET,  "crypt",  MAX_SEQ_LEN)
        replay = build_examples_from_parquet(tok, REPLAY_PARQUET, "replay", MAX_SEQ_LEN)
        if REPLAY_SUBSAMPLE is not None and len(replay) > REPLAY_SUBSAMPLE:
            import random as _rng
            _rng.seed(SEED)
            _keys = _rng.sample(list(replay.keys()), REPLAY_SUBSAMPLE)
            replay = {k: replay[k] for k in _keys}
            print(f"  replay subsampled: {len(replay)} rows (REPLAY_SUBSAMPLE={REPLAY_SUBSAMPLE})")
        examples_by_id = {**crypt, **replay}

        print(f"  crypt : {len(crypt)}  rows  ({CRYPT_PARQUET})")
        print(f"  replay: {len(replay)} rows  ({REPLAY_PARQUET})")

        pool = list(crypt.keys()) * CRYPT_EPOCHS + list(replay.keys()) * REPLAY_EPOCHS
        print(
            f"  pool  : {len(pool)} ids  "
            f"(crypt x{CRYPT_EPOCHS}={len(crypt)*CRYPT_EPOCHS} + "
            f"replay x{REPLAY_EPOCHS}={len(replay)*REPLAY_EPOCHS})"
        )
        schedule = build_flat_schedule(pool, BATCH_SIZE, NUM_STEPS, SEED)
        print(f"  schedule: {len(schedule)} steps  (seed={SEED})")

        _missing = {pid for ids in schedule for pid in ids} - set(examples_by_id)
        assert not _missing, f"schedule references {len(_missing)} unknown ids: {list(_missing)[:3]}"

        total_unmasked = sum(sum(e["weights"]) for e in examples_by_id.values())
        total_tokens   = sum(len(e["tokens"])  for e in examples_by_id.values())
        print(f"Tokenised {len(examples_by_id)} unique examples, {total_tokens:,} tokens "
              f"(unmasked={total_unmasked:,.0f})")

    # ── Load base model ───────────────────────────────────────────────
    gc.collect(); torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH, max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False, load_in_8bit=False, full_finetuning=False,
        trust_remote_code=True, unsloth_force_compile=True,
        attn_implementation="eager", dtype=torch.bfloat16,
    )
    if IS_MODAL_WORKER:
        hf_cache_vol.commit()  # noqa: F821

    # ── Wrap in LoRA ──────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model, r=LORA_RANK, target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )
    FastLanguageModel.for_training(model)

    # ── Patch Mamba CUDA fast path ────────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m; break
    assert nemotron_mod is not None, "Could not find modeling_nemotron_h module"
    print(f"is_fast_path_available was: {nemotron_mod.is_fast_path_available}")
    nemotron_mod.is_fast_path_available = True  # type: ignore[attr-defined]
    print("Patched is_fast_path_available = True")

    # ── Manually add lm_head LoRA (Unsloth drops it for MoE) ─────────
    _causal_lm = model
    while hasattr(_causal_lm, "model"):
        _causal_lm = _causal_lm.model
    _lm_head = _causal_lm.lm_head
    if not isinstance(_lm_head, LoraLinear):
        _cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            _cfg, "default", target=_lm_head, target_name="lm_head", parent=_causal_lm,
        )
        print("Manually added LoRA to lm_head")
    else:
        print("lm_head already has LoRA")

    # ── Cast LoRA params to fp32 (base stays bf16 except MoE router) ─
    for name, param in model.named_parameters():
        if ".lora_" in name:
            param.data = param.data.to(torch.float32)

    for name, param in model.named_parameters():
        if ".lora_" in name:
            assert param.dtype == torch.float32, f"LoRA {name}: expected fp32"
            continue
        if ".mixer.gate." in name:
            assert param.dtype == torch.float32, f"{name}: expected fp32 (MoE router)"
            continue
        assert param.dtype == torch.bfloat16, f"{name}: expected bf16"
    print("Verified: LoRA params fp32, base params bf16 (MoE router fp32)")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total parameters")

    # ── Patch forward with Cut Cross-Entropy ─────────────────────────
    _base = model
    while hasattr(_base, "model"):
        _base = _base.model

    def _patched_causal_forward(input_ids=None, attention_mask=None, labels=None, **kwargs):
        backbone_out = _base.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            **{k: v for k, v in kwargs.items() if k in ("position_ids", "past_key_values", "use_cache")},
        )
        hidden_states = backbone_out[0]
        lm_head   = _base.lm_head
        lm_weight = lm_head.base_layer.weight + lm_head.scaling["default"] * lm_head.lora_B["default"].weight @ lm_head.lora_A["default"].weight
        if labels is not None:
            per_token_ce = linear_cross_entropy(hidden_states, lm_weight, labels, reduction="none")
            loss = per_token_ce.mean()
        else:
            per_token_ce = None; loss = None
        model._cached_per_token_ce = per_token_ce  # type: ignore[attr-defined]
        return loss

    _base.forward = _patched_causal_forward
    print("Patched CausalLM.forward with CCE (no logits materialization)")

    # ── Load adapter weights (EXPERIMENT="continue" only) ────────────
    if RESET_WEIGHTS:
        print("RESET_WEIGHTS=True (scratch) — fresh LoRA init, no adapter loaded")
        loaded = 0; adapter_weights: dict = {}
    else:
        print(f"Loading adapter from {ADAPTER_SRC}...")
        from peft import load_peft_weights
        adapter_weights = load_peft_weights(ADAPTER_SRC)
        model_sd = model.state_dict()
        new_sd: dict = {}; loaded = 0
        for ak, av in adapter_weights.items():
            if ak in model_sd:
                new_sd[ak] = av; loaded += 1; continue
            ak_d = ak.replace(".lora_A.weight", ".lora_A.default.weight").replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_d in model_sd:
                new_sd[ak_d] = av; loaded += 1; continue
            ak_lm = ak.replace(".backbone.lm_head.", ".lm_head.")
            ak_lm_d = ak_lm.replace(".lora_A.weight", ".lora_A.default.weight").replace(".lora_B.weight", ".lora_B.default.weight")
            if ak_lm_d in model_sd:
                new_sd[ak_lm_d] = av; loaded += 1; continue
        model.load_state_dict(new_sd, strict=False)
        assert loaded == len(adapter_weights), f"Not all adapter weights loaded: {loaded}/{len(adapter_weights)}"
        print(f"  Loaded {loaded}/{len(adapter_weights)} weights into model")

    # ── IN_PROJ_ONLY freeze ───────────────────────────────────────────
    print(f"{IN_PROJ_ONLY=}")
    if IN_PROJ_ONLY:
        for name, param in model.named_parameters():
            if param.requires_grad and ".in_proj." not in name:
                param.requires_grad = False
    print(f"  {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable"
          f" / {sum(p.numel() for p in model.parameters() if not p.requires_grad):,} frozen")

    # ── MoE tied-weight params (Tinker convention) ────────────────────
    moe_tied_params: list[torch.Tensor] = []
    if MOE_TIE_WEIGHTS:
        w1_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_names = ("down_proj", ".w2.")
        for name, param in model.named_parameters():
            if not param.requires_grad or ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(p in name for p in w1_names)
            is_w2 = any(p in name for p in w2_names)
            is_A  = ".lora_A." in name
            is_B  = ".lora_B." in name
            if not ((is_w1 and is_A) or (is_w2 and is_B)):
                continue
            if param.dim() < 2 or param.shape[0] <= 1:
                continue
            moe_tied_params.append(param)

        def _tie_param_init():
            with torch.no_grad():
                for p in moe_tied_params:
                    p.data.copy_(p.data.mean(dim=0, keepdim=True).expand_as(p.data))

        def _tie_grads():
            with torch.no_grad():
                for p in moe_tied_params:
                    if p.grad is None: continue
                    p.grad.copy_(p.grad.sum(dim=0, keepdim=True).expand_as(p.grad))

        print(f"MoE weight tying: {len(moe_tied_params)} params identified for tying")
        if moe_tied_params:
            print(f"  example shapes: {[tuple(p.shape) for p in moe_tied_params[:3]]}")
        _tie_param_init()
    else:
        def _tie_grads(): pass

    # ── Training loop ─────────────────────────────────────────────────
    gc.collect(); torch.cuda.empty_cache()

    if SKIP_TRAINING:
        print("SKIP_TRAINING=True — no optimizer steps; saving fresh LoRA init as baseline")
        training_log: list[str] = ["SKIP_TRAINING=True: 0 steps, fresh LoRA init (zero-delta adapter)"]
    else:
        device = next(model.parameters()).device
        optimizer: torch.optim.AdamW | None = None
        training_log: list[str] = []

        def _log(msg):
            print(msg, flush=True); training_log.append(msg)

        num_steps = len(schedule)
        if num_steps == 0:
            _log("WARNING: empty schedule — nothing to train on.")
        _log(
            f"Training: {num_steps} steps, batch_size={BATCH_SIZE}, "
            f"micro_batch_size={MICRO_BATCH_SIZE}, lr={LEARNING_RATE} "
            f"(flat shuffle, EXPERIMENT={EXPERIMENT!r})"
        )

        for step in range(num_steps):
            batch         = [examples_by_id[pid] for pid in schedule[step]]
            batch_tokens  = [e["tokens"]  for e in batch]
            batch_targets = [e["targets"] for e in batch]
            batch_weights = [e["weights"] for e in batch]

            n       = len(batch)
            n_accum = math.ceil(n / MICRO_BATCH_SIZE)
            total_loss_sum = total_weight_sum = 0.0

            for mb_start in range(0, n, MICRO_BATCH_SIZE):
                mb_end  = min(mb_start + MICRO_BATCH_SIZE, n)
                mb_toks = batch_tokens[mb_start:mb_end]
                mb_tgts = batch_targets[mb_start:mb_end]
                mb_wts  = batch_weights[mb_start:mb_end]

                n_micro   = len(mb_toks)
                max_len   = max(len(t) for t in mb_toks)
                total_len = sum(len(t) for t in mb_toks)

                padded_input   = torch.zeros(n_micro, max_len, dtype=torch.long,    device=device)
                padded_targets = torch.zeros(n_micro, max_len, dtype=torch.long,    device=device)
                padded_weights = torch.zeros(n_micro, max_len, dtype=torch.float32, device=device)
                attention_mask = torch.zeros(n_micro, max_len, dtype=torch.long,    device=device)
                for i in range(n_micro):
                    sl = len(mb_toks[i])
                    padded_input[i,   :sl] = torch.tensor(mb_toks[i], dtype=torch.long)
                    padded_targets[i, :sl] = torch.tensor(mb_tgts[i], dtype=torch.long)
                    padded_weights[i, :sl] = torch.tensor(mb_wts[i],  dtype=torch.float32)
                    attention_mask[i, :sl] = 1

                t0 = time.time()
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    model(input_ids=padded_input, attention_mask=attention_mask,
                          labels=padded_targets, use_cache=False)
                    per_token_ce  = model._cached_per_token_ce  # type: ignore[attr-defined]
                    weight_sum_t  = padded_weights.sum()
                    loss_sum_t    = (per_token_ce * padded_weights).sum()
                    loss = loss_sum_t / weight_sum_t if weight_sum_t > 0 else loss_sum_t * 0.0

                (loss / n_accum).backward()
                total_loss_sum   += loss_sum_t.item()
                total_weight_sum += weight_sum_t.item()
                del loss, per_token_ce

                t1 = time.time()
                print(
                    f"    micro-batch {mb_start//MICRO_BATCH_SIZE}: {n_micro} seqs, "
                    f"max_len={max_len}, total_len={total_len}, wall={t1-t0:.1f}s, "
                    f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB, "
                    f"mem={torch.cuda.memory_allocated()/1e9:.1f}GB"
                )

            if optimizer is None:
                optimizer = torch.optim.AdamW(
                    [p for p in model.parameters() if p.requires_grad],
                    lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
                )
            lr = LEARNING_RATE * (1 - step / num_steps)
            for pg in optimizer.param_groups:
                pg["lr"] = lr
            _tie_grads()
            grad_norm = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1e9
            )
            optimizer.step(); optimizer.zero_grad()
            loss_mean = total_loss_sum / total_weight_sum if total_weight_sum > 0 else 0
            _log(f"  step {step+1}/{num_steps}: loss:mean={loss_mean:.6f}, grad_norm={grad_norm:.4f}, lr={lr:.2e}")

        print(f"\nTraining complete. Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    # ── Save adapter + rename lm_head keys ───────────────────────────
    from safetensors.torch import load_file, save_file
    save_dir = "." if IS_KAGGLE else OUTPUT_DIR
    os.makedirs(save_dir, exist_ok=True)
    for _f in os.listdir(save_dir):
        if _f.startswith("adapter"):
            os.remove(os.path.join(save_dir, _f))
    model.save_pretrained(save_dir)
    st_path = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st_path)
    save_file(
        {k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
         for k, v in tensors.items()},
        st_path,
    )

    _ucache = "unsloth_compiled_cache"
    if os.path.isdir(_ucache):
        import shutil as _sh; _sh.rmtree(_ucache)

    # ── Package & ship ────────────────────────────────────────────────
    if IS_KAGGLE:
        import zipfile
        adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
        SUBMISSION_ZIP = "submission.zip"
        with zipfile.ZipFile(SUBMISSION_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
            for fname in adapter_files:
                zf.write(os.path.join(save_dir, fname), fname)
        for fname in adapter_files:
            os.remove(os.path.join(save_dir, fname))
        print(f"Wrote {SUBMISSION_ZIP}")
    else:  # IS_MODAL_WORKER
        import shutil, tempfile
        with open(os.path.join(save_dir, "training_log.txt"), "w") as f:
            f.write("\n".join(training_log) + "\n")
        output_vol.commit()  # noqa: F821
        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(os.path.join(kaggle_dir, "access_token"), "w") as f:
            f.write(os.environ["KAGGLE_API_TOKEN"])
        upload_dir = tempfile.mkdtemp()
        for fname in os.listdir(save_dir):
            shutil.copy(os.path.join(save_dir, fname), upload_dir)
        with open(os.path.join(upload_dir, "dataset-metadata.json"), "w") as f:
            json.dump({"id": KAGGLE_DATASET, "title": KAGGLE_DATASET.split("/")[1]}, f)
        print(f"Uploading to Kaggle {KAGGLE_DATASET}...")
        subprocess.run(
            ["kaggle", "datasets", "version", "-p", upload_dir,
             "-m", f"flat-shuffle EXPERIMENT={EXPERIMENT}"],
            check=True,
        )
        print("Kaggle upload complete.")
    print("Training complete.")
    return training_log

In [ ]:
# ── Modal glue: image, app, volumes, train_remote, main ──────────────
if not IS_KAGGLE:
    import modal

    train_image = (
        modal.Image.from_registry("nvidia/cuda:12.8.1-devel-ubuntu22.04", add_python="3.12")
        .entrypoint([])
        .apt_install("git", "build-essential", "clang")
        .pip_install("torch==2.10.0", extra_index_url="https://download.pytorch.org/whl/cu128")
        .pip_install(
            "safetensors>=0.5.0", "transformers>=4.56.2", "accelerate>=1.0.0",
            "peft>=0.15.0", "bitsandbytes>=0.45.0", "huggingface_hub>=0.36.2",
            "hf-transfer>=0.1.9", "numpy", "pillow", "torchvision", "datasets",
            "sentencepiece", "xformers", "cut-cross-entropy>=25.1.0",
            "wheel", "setuptools", "trl", "kaggle>=1.6.0",
        )
        .run_commands(
            'python -c "import torch.utils.cpp_extension as e; p=e.__file__; t=open(p).read().replace(\'raise RuntimeError(CUDA_MISMATCH_MESSAGE\', \'pass  # \'); open(p,\'w\').write(t)"',
            "TORCH_CUDA_ARCH_LIST='12.0' pip wheel --no-build-isolation --wheel-dir /wheels mamba_ssm==2.3.1 causal_conv1d==1.6.1",
            "pip install --no-deps /wheels/mamba_ssm-*.whl /wheels/causal_conv1d-*.whl",
            "pip install --no-deps 'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo'",
            "pip install --no-deps 'unsloth[base] @ git+https://github.com/unslothai/unsloth'",
        )
        .pip_install("einops")
        .env({"HF_HOME": "/root/.cache/huggingface"})
    )

    hf_cache_vol = modal.Volume.from_name("huggingface-cache",    create_if_missing=True)
    merged_vol   = modal.Volume.from_name("merged-adapter",       create_if_missing=True)
    corpus_vol   = modal.Volume.from_name("corpus-data",          create_if_missing=True)
    output_vol   = modal.Volume.from_name("post-finetune-output", create_if_missing=True)

    app = modal.App("post-finetune-flat-shuffle")

    @app.function(
        image=train_image,
        gpu="RTX-PRO-6000",
        volumes={
            "/root/.cache/huggingface": hf_cache_vol,
            "/merged": merged_vol,
            "/data":   corpus_vol,
            "/output": output_vol,
        },
        timeout=8 * 60 * MINUTES,
        secrets=[modal.Secret.from_local_environ(["KAGGLE_API_TOKEN"])],
    )
    def train_remote() -> None:
        run_training()

    if IS_MODAL_LAUNCHER:
        @app.local_entrypoint()
        def main() -> None:
            train_remote.remote()

In [ ]:
# On Kaggle, trigger training directly after cells load.
# On Modal worker, Modal's runtime calls train_remote() -> run_training().
# On Modal launcher, main() submits the remote call instead.
if IS_KAGGLE:
    _training_log = run_training()

In [ ]:
import re
import matplotlib.pyplot as plt

log_lines = []
if IS_KAGGLE and "_training_log" in dir():
    log_lines = _training_log
elif os.path.exists("training_log.txt"):
    with open("training_log.txt") as f:
        log_lines = f.readlines()

steps, losses = [], []
for line in log_lines:
    m = re.search(r"step (\d+)/\d+: loss:mean=([\d.]+)", line)
    if m:
        steps.append(int(m.group(1)))
        losses.append(float(m.group(2)))

if steps:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses, linewidth=1.2)
    ax.set_xlabel("Step")
    ax.set_ylabel("Loss (weighted mean)")
    ax.set_title(f"Training Loss — EXPERIMENT={EXPERIMENT!r}")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("No training loss data found in log.")